# Mercedes F1 Infringement Document Filter

This notebook extracts Mercedes-specific infringement documents from the FIA PDF collection and converts them to text files for further processing.

## Objective
- Parse all PDF files in the Documents folder
- Filter for documents addressed to "Mercedes-AMG PETRONAS F1 Team"
- Extract text content from filtered PDFs
- Save as individual .txt files organized by year


In [28]:
pip install PyPDF2

In [29]:
# Import required libraries
import os
import re
from pathlib import Path
import PyPDF2
import pandas as pd
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


In [30]:
# Updated configuration for new folder structure
base_path = Path("Documents")
years = ["2020_inf_profile", "2021_inf_profile", 
         "2022_inf_profile", "2023_inf_profile", "2024_inf_profile"]

print("Updated folder structure:")
for year in years:
    year_path = base_path / year
    if year_path.exists():
        pdf_count = len(list(year_path.glob("*.pdf")))
        txt_count = len(list(year_path.glob("*.txt")))
        print(f"  {year}: {pdf_count} PDF files, {txt_count} TXT files")
    else:
        print(f"  {year}: Folder not found")

# Override the original years variable
years = ["2020_inf_profile", "2021_inf_profile", 
         "2022_inf_profile", "2023_inf_profile", "2024_inf_profile"]


Updated folder structure:
  2020_inf_profile: 131 PDF files, 13 TXT files
  2021_inf_profile: 175 PDF files, 17 TXT files
  2022_inf_profile: 227 PDF files, 17 TXT files
  2023_inf_profile: 194 PDF files, 23 TXT files
  2024_inf_profile: 215 PDF files, 36 TXT files


In [31]:
# Define paths
base_path = Path("Documents")
years = ["2020-infirdgement_profile", "2021-infridgement_profile", 
         "2022-infridgement_profile", "2023-infridgement_profile"]

print("Available year folders:")
for year in years:
    year_path = base_path / year
    if year_path.exists():
        pdf_count = len(list(year_path.glob("*.pdf")))
        print(f"  {year}: {pdf_count} PDF files")


Available year folders:


In [32]:
# Function to extract text from PDF
def extract_pdf_text(pdf_path):
    """
    Extract text from PDF file
    Returns the full text content
    """
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""
            
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                text += page.extract_text() + "\n"
            
            return text
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return None


In [46]:
# Updated function to check if document is addressed to Mercedes (includes car numbers)
def is_mercedes_document_updated(text):
    """
    Check if the document is addressed to Mercedes team or involves Mercedes drivers
    Look for the specific header pattern and car numbers
    """
    if not text:
        return False
    
    # Convert to lowercase for case-insensitive matching
    text_lower = text.lower()
    
    # Look for Mercedes team references in the header section
    mercedes_patterns = [
        r'to:\s*the team manager,?\s*mercedes[-\s]*amg\s*petronas\s*f1\s*team',
        r'mercedes[-\s]*amg\s*petronas\s*f1\s*team',
        r'mercedes\s*petronas\s*f1\s*team',
        r'mercedes\s*amg\s*petronas'
    ]
    
    # Mercedes car numbers by year:
    # 2020-2021: Car 44 (Hamilton), Car 77 (Bottas)
    # 2022-2023: Car 44 (Hamilton), Car 63 (Russell)
    mercedes_car_numbers = [44, 77, 63]
    
    # Check first 2000 characters (header section)
    header_section = text_lower[:2000]
    
    # Check for Mercedes team references
    for pattern in mercedes_patterns:
        if re.search(pattern, header_section, re.IGNORECASE):
            return True
    
    # Check for Mercedes car numbers in document title/header
    # Look for patterns like "Car 44", "Car 77", etc.
    for car_num in mercedes_car_numbers:
        car_pattern = rf'\bcar\s*{car_num}\b'
        if re.search(car_pattern, header_section, re.IGNORECASE):
            return True
    
    return False

# Use the updated function instead of the original
is_mercedes_document = is_mercedes_document_updated


In [34]:
# Function to clean and save text content
def clean_text(text):
    """
    Clean extracted text
    """
    if not text:
        return ""
    
    # Remove excessive whitespace and normalize
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text


In [35]:
# Updated configuration for new folder structure
base_path = Path("Documents")
years = ["2020_inf_profile", "2021_inf_profile", 
         "2022_inf_profile", "2023_inf_profile", "2024_inf_profile"]

print("Updated folder structure:")
for year in years:
    year_path = base_path / year
    if year_path.exists():
        pdf_count = len(list(year_path.glob("*.pdf")))
        txt_count = len(list(year_path.glob("*.txt")))
        print(f"  {year}: {pdf_count} PDF files, {txt_count} TXT files")
    else:
        print(f"  {year}: Folder not found")

# Override the original years variable
years = ["2020_inf_profile", "2021_inf_profile", 
         "2022_inf_profile", "2023_inf_profile", "2024_inf_profile"]


Updated folder structure:
  2020_inf_profile: 131 PDF files, 13 TXT files
  2021_inf_profile: 175 PDF files, 17 TXT files
  2022_inf_profile: 227 PDF files, 17 TXT files
  2023_inf_profile: 194 PDF files, 23 TXT files
  2024_inf_profile: 215 PDF files, 36 TXT files


In [ ]:
# Updated function to check if document is addressed to Mercedes (includes car numbers)
def is_mercedes_document_updated(text):
    """
    Check if the document is addressed to Mercedes team or involves Mercedes drivers
    Look for the specific header pattern and car numbers
    EXCLUDE documents addressed to "All Teams, All Officials"
    """
    if not text:
        return False
    
    # Convert to lowercase for case-insensitive matching
    text_lower = text.lower()
    
    # Check first 2000 characters (header section)
    header_section = text_lower[:2000]
    
    # EXCLUDE documents addressed to "All Teams, All Officials"
    exclude_patterns = [
        r'to\s+all\s+teams,?\s+all\s+officials',
        r'to\s+all\s+teams',
        r'all\s+teams,?\s+all\s+officials',
        r'to\s+all\s+competitors',
        r'all\s+competitors'
    ]
    
    for pattern in exclude_patterns:
        if re.search(pattern, header_section, re.IGNORECASE):
            return False  # Exclude this document
    
    # Look for Mercedes team references in the header section
    mercedes_patterns = [
        r'to:\s*the team manager,?\s*mercedes[-\s]*amg\s*petronas\s*f1\s*team',
        r'mercedes[-\s]*amg\s*petronas\s*f1\s*team',
        r'mercedes\s*petronas\s*f1\s*team',
        r'mercedes\s*amg\s*petronas'
    ]
    
    # Mercedes car numbers by year:
    # 2020-2021: Car 44 (Hamilton), Car 77 (Bottas)
    # 2022-2023: Car 44 (Hamilton), Car 63 (Russell)
  
    mercedes_car_numbers = [44, 77, 63]
    
    # Check for Mercedes team references
    for pattern in mercedes_patterns:
        if re.search(pattern, header_section, re.IGNORECASE):
            return True
    
    # Check for Mercedes car numbers in document title/header
    # Look for patterns like "Car 44", "Car 77", etc.
    for car_num in mercedes_car_numbers:
        car_pattern = rf'\bcar\s*{car_num}\b'
        if re.search(car_pattern, header_section, re.IGNORECASE):
            return True
    
    return False

# Use the updated function instead of the original
is_mercedes_document = is_mercedes_document_updated


: 

In [37]:
# Updated main processing function to handle existing TXT files
def process_year_folder_updated(year_folder):
    """
    Process all PDFs in a year folder and extract Mercedes documents
    Skip PDFs that already have corresponding TXT files
    """
    year_path = base_path / year_folder
    if not year_path.exists():
        print(f"Folder {year_path} does not exist")
        return
    
    pdf_files = list(year_path.glob("*.pdf"))
    print(f"\nProcessing {len(pdf_files)} PDF files in {year_folder}...")
    
    mercedes_docs = []
    skipped_count = 0
    
    for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
        # Check if TXT file already exists
        txt_filename = pdf_file.stem + ".txt"
        txt_path = year_path / txt_filename
        
        if txt_path.exists():
            # TXT file already exists, skip processing
            skipped_count += 1
            continue
        
        # Extract text from PDF
        text = extract_pdf_text(pdf_file)
        
        if text and is_mercedes_document(text):
            # Clean the text
            cleaned_text = clean_text(text)
            
            # Save as text file
            try:
                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)
                
                mercedes_docs.append({
                    'pdf_file': pdf_file.name,
                    'txt_file': txt_filename,
                    'year': year_folder,
                    'text_length': len(cleaned_text)
                })
                
                print(f"✓ Found Mercedes document: {pdf_file.name}")
                
            except Exception as e:
                print(f"Error saving {txt_filename}: {e}")
    
    print(f"Skipped {skipped_count} PDFs (TXT files already exist)")
    return mercedes_docs

# Use the updated function
process_year_folder = process_year_folder_updated


In [38]:
# Function to check if document is addressed to Mercedes
def is_mercedes_document(text):
    """
    Check if the document is addressed to Mercedes team
    Look for the specific header pattern
    """
    if not text:
        return False
    
    # Convert to lowercase for case-insensitive matching
    text_lower = text.lower()
    
    # Look for Mercedes team references in the header section
    mercedes_patterns = [
        r'to:\s*the team manager,?\s*mercedes[-\s]*amg\s*petronas\s*f1\s*team',
        r'mercedes[-\s]*amg\s*petronas\s*f1\s*team',
        r'mercedes\s*petronas\s*f1\s*team',
        r'mercedes\s*amg\s*petronas'
    ]
    
    # Check first 2000 characters (header section)
    header_section = text_lower[:2000]
    
    for pattern in mercedes_patterns:
        if re.search(pattern, header_section, re.IGNORECASE):
            return True
    
    return False


In [39]:
# Updated main processing function to handle existing TXT files
def process_year_folder_updated(year_folder):
    """
    Process all PDFs in a year folder and extract Mercedes documents
    Skip PDFs that already have corresponding TXT files
    """
    year_path = base_path / year_folder
    if not year_path.exists():
        print(f"Folder {year_path} does not exist")
        return
    
    pdf_files = list(year_path.glob("*.pdf"))
    print(f"\nProcessing {len(pdf_files)} PDF files in {year_folder}...")
    
    mercedes_docs = []
    skipped_count = 0
    
    for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
        # Check if TXT file already exists
        txt_filename = pdf_file.stem + ".txt"
        txt_path = year_path / txt_filename
        
        if txt_path.exists():
            # TXT file already exists, skip processing
            skipped_count += 1
            continue
        
        # Extract text from PDF
        text = extract_pdf_text(pdf_file)
        
        if text and is_mercedes_document(text):
            # Clean the text
            cleaned_text = clean_text(text)
            
            # Save as text file
            try:
                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)
                
                mercedes_docs.append({
                    'pdf_file': pdf_file.name,
                    'txt_file': txt_filename,
                    'year': year_folder,
                    'text_length': len(cleaned_text)
                })
                
                print(f"✓ Found Mercedes document: {pdf_file.name}")
                
            except Exception as e:
                print(f"Error saving {txt_filename}: {e}")
    
    print(f"Skipped {skipped_count} PDFs (TXT files already exist)")
    return mercedes_docs

# Use the updated function
process_year_folder = process_year_folder_updated


In [40]:
# Focus on 2024 processing (new data)
print("="*60)
print("PROCESSING 2024 INF_PROFILE (NEW DATA)")
print("="*60)

# Process only 2024 folder
year_2024 = "2024_inf_profile"
mercedes_2024_docs = process_year_folder(year_2024)

if mercedes_2024_docs:
    print(f"\nFound {len(mercedes_2024_docs)} NEW Mercedes documents in 2024")
    for doc in mercedes_2024_docs:
        print(f"  - {doc['pdf_file']} ({doc['text_length']} chars)")
else:
    print("\nNo NEW Mercedes documents found in 2024")


PROCESSING 2024 INF_PROFILE (NEW DATA)

Processing 215 PDF files in 2024_inf_profile...


Processing PDFs: 100%|██████████| 215/215 [00:05<00:00, 36.83it/s]

Skipped 36 PDFs (TXT files already exist)

No NEW Mercedes documents found in 2024


In [41]:
# Function to clean and save text content
def clean_text(text):
    """
    Clean extracted text
    """
    if not text:
        return ""
    
    # Remove excessive whitespace and normalize
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text


In [42]:
# Main processing function
def process_year_folder(year_folder):
    """
    Process all PDFs in a year folder and extract Mercedes documents
    """
    year_path = base_path / year_folder
    if not year_path.exists():
        print(f"Folder {year_path} does not exist")
        return
    
    pdf_files = list(year_path.glob("*.pdf"))
    print(f"\nProcessing {len(pdf_files)} PDF files in {year_folder}...")
    
    mercedes_docs = []
    
    for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
        # Extract text from PDF
        text = extract_pdf_text(pdf_file)
        
        if text and is_mercedes_document(text):
            # Clean the text
            cleaned_text = clean_text(text)
            
            # Create output filename
            txt_filename = pdf_file.stem + ".txt"
            txt_path = year_path / txt_filename
            
            # Save as text file
            try:
                with open(txt_path, 'w', encoding='utf-8') as f:
                    f.write(cleaned_text)
                
                mercedes_docs.append({
                    'pdf_file': pdf_file.name,
                    'txt_file': txt_filename,
                    'year': year_folder,
                    'text_length': len(cleaned_text)
                })
                
                print(f"✓ Found Mercedes document: {pdf_file.name}")
                
            except Exception as e:
                print(f"Error saving {txt_filename}: {e}")
    
    return mercedes_docs


In [43]:
# Process all year folders
all_mercedes_docs = []

for year_folder in years:
    mercedes_docs = process_year_folder(year_folder)
    if mercedes_docs:
        all_mercedes_docs.extend(mercedes_docs)
        print(f"Found {len(mercedes_docs)} Mercedes documents in {year_folder}")
    else:
        print(f"No Mercedes documents found in {year_folder}")



Processing 131 PDF files in 2020_inf_profile...


Processing PDFs:   2%|▏         | 3/131 [00:00<00:04, 27.18it/s]

Processing PDFs:   5%|▌         | 7/131 [00:00<00:13,  9.08it/s]

✓ Found Mercedes document: 2020 Austrian Grand Prix - Decision - Car 44 - alleged failure to slow for yellow flags.pdf


Processing PDFs:  10%|▉         | 13/131 [00:01<00:10, 11.00it/s]

...

✓ Found Mercedes document: 2024 United States Grand Prix - Infringement - Car 63 - Breach of Parc Ferme.pdf
✓ Found Mercedes document: 2024 United States Grand Prix - Infringement - Car 63 - Forcing another driver off the track.pdf


Processing PDFs:  99%|█████████▊| 212/215 [00:20<00:00, 26.75it/s]

✓ Found Mercedes document: 2024 United States Grand Prix - Infringement - Qualifying Deleted Lap Times.pdf
✓ Found Mercedes document: 2024 United States Grand Prix - Infringement - Race Deleted Lap Times.pdf


Processing PDFs: 100%|██████████| 215/215 [00:20<00:00, 10.27it/s]

✓ Found Mercedes document: 2024 United States Grand Prix - Infringement - Sprint Deleted Lap Times.pdf
Found 36 Mercedes documents in 2024_inf_profile


In [44]:
# Summary statistics
if all_mercedes_docs:
    df_mercedes = pd.DataFrame(all_mercedes_docs)
    
    print("\n" + "="*50)
    print("MERCEDES DOCUMENT EXTRACTION SUMMARY")
    print("="*50)
    
    print(f"Total Mercedes documents found: {len(all_mercedes_docs)}")
    print(f"Total text length: {sum([doc['text_length'] for doc in all_mercedes_docs]):,} characters")
    
    print("\nDocuments by year:")
    year_counts = df_mercedes['year'].value_counts().sort_index()
    for year, count in year_counts.items():
        print(f"  {year}: {count} documents")
    
    print("\nSample documents:")
    for _, row in df_mercedes.head().iterrows():
        print(f"  - {row['pdf_file']} ({row['text_length']} chars)")
    
    # Save summary
    df_mercedes.to_csv('mercedes_documents_summary.csv', index=False)
    print("\nSummary saved to: mercedes_documents_summary.csv")
    
else:
    print("\nNo Mercedes documents found. Please check the filtering criteria.")



MERCEDES DOCUMENT EXTRACTION SUMMARY
Total Mercedes documents found: 106
Total text length: 190,742 characters

Documents by year:
  2020_inf_profile: 13 documents
  2021_inf_profile: 17 documents
  2022_inf_profile: 17 documents
  2023_inf_profile: 23 documents
  2024_inf_profile: 36 documents

Sample documents:
  - 2020 Austrian Grand Prix - Decision - Car 44 - alleged failure to slow for yellow flags.pdf (1438 chars)
  - 2020 Austrian Grand Prix - Decision - review of decision (document 33).pdf (1124 chars)
  - 2020 Austrian Grand Prix - Offence - Car 44 - Failure to slow for yellow flags (post review).pdf (1632 chars)
  - 2020 Austrian Grand Prix - Offence - Car 44 - incident with car 23.pdf (1418 chars)
  - 2020 Austrian Grand Prix - Offence - Car 44 - Leaving the track in turn 10.pdf (1623 chars)

Summary saved to: mercedes_documents_summary.csv


In [45]:
# Display sample of extracted text for verification
if all_mercedes_docs:
    print("\n" + "="*50)
    print("SAMPLE EXTRACTED TEXT")
    print("="*50)
    
    # Get first Mercedes document
    first_doc = all_mercedes_docs[0]
    year_path = base_path / first_doc['year']
    txt_path = year_path / first_doc['txt_file']
    
    if txt_path.exists():
        with open(txt_path, 'r', encoding='utf-8') as f:
            sample_text = f.read()
        
        print(f"Document: {first_doc['pdf_file']}")
        print(f"Year: {first_doc['year']}")
        print(f"Length: {len(sample_text)} characters")
        print("\nFirst 1000 characters:")
        print("-" * 50)
        print(sample_text[:1000] + "..." if len(sample_text) > 1000 else sample_text)



SAMPLE EXTRACTED TEXT
Document: 2020 Austrian Grand Prix - Decision - Car 44 - alleged failure to slow for yellow flags.pdf
Year: 2020_inf_profile
Length: 1438 characters

First 1000 characters:
--------------------------------------------------
From The Stewards To The Team Manager, Mercedes-AMG Petronas F1 TeamDocument 33 Date 04 July 2020 Time 19:44 2020 AUSTRIAN GRAND PRIX 2 - 5 July 2020 The StewardsThe Stewards, having received a report from the Race Director, summoned (document 29) and heard from the driver and team representative, have considered the following matter and determine the following: No / Driver 44 - Lewis Hamilton Competitor Mercedes-AMG Petronas F1 Team Time 15:59 Session Qualifying Fact Alleged failure to slow for single waved yellow flags between turn 5 and 7. Offence Alleged breach of Appendix H Article 2.5.5.1.b) of the FIA International Sporting Code. Decision No further action. Reason The Stewards heard from the driver of Car 44 (Lewis Hamilton) and the tea

## Next Steps

After running this notebook:
1. Review the extracted Mercedes documents
2. Check the sample text to ensure proper extraction
3. Proceed to consolidation and preprocessing steps
4. Begin text summarization process
